# SCAF Checkpoint Causal Leak Analysis
# Fock-PARFLM v2.1 — Comprehensive Post-Training Audit

Multi-phase causal analysis of a trained Fock-PARFLM checkpoint using the
**SCAF (SemSimula Causal Auditing Framework)** library.

## Phases

1. **Phase 1 — SCAF Full Audit**: Controls + future perturbation + target
   relocation + mediation attribution.
2. **Phase 2 — Interventional Leak Frame**: Build a tidy counterfactual
   dataset and compute exact ATE with paired inference.
3. **Phase 3 — Trained-Scale Leak Probe**: Legacy probe (future perturbation
   at trained scale + honest vs standard PPL).
4. **Phase 4 — Register Diagnostics**: Routing quality (entropy, diversity,
   alpha\_max) per layer.
5. **Phase 5 — Visualisation Dashboard**: 6-panel figure summarising all
   findings.
6. **Phase 6 — DoWhy/EconML Estimands** *(optional)*: Formal ATE, CATE by
   distance-to-cut, refutation tests.

All results are persisted to a configurable GDrive location.


In [ ]:
# ── Cell 0: Configuration ──────────────────────────────────────────
# ── Checkpoint to analyse ──
CHECKPOINT_PATH = '/content/drive/MyDrive/semsimula_fock_multixi_structured_vtheta/A2/seed0/ckpt_best.pt'

# ── Output directory (GDrive) ──
OUTPUT_DIR = '/content/drive/MyDrive/scaf_analysis_results'

# ── Model family (auto-detected if checkpoint has 'model_cfg') ──
#    'auto'            → read from checkpoint['model_cfg'] /
#                        ['v_theta_variant'] / ['v_theta_kind']
#    'mlp'             → MLP V_theta (ScalarPotentialMultiXi)
#    'gaussian'        → Gaussian V_theta, ambiguous isotropic/anisotropic —
#                        resolved via GAUSSIAN_VARIANT below (default: aniso)
#    'gaussian_aniso'  → AnisotropicDepthConditionedGaussianVTheta (explicit)
#    'gaussian_iso'    → DepthConditionedMultiContextGaussianVTheta (explicit)
#    'sq3'             → MixtureQuadraticVTheta via StructuredVThetaMultiXiAdapter
MODEL_FAMILY = 'auto'

# ── Which Gaussian V_theta implementation to build when the detected/
#    configured family is the ambiguous 'gaussian' tag. Most checkpoints
#    worth debugging at d=384/d=768 scale use the anisotropic (low-rank
#    precision) variant, so it is the default. Set to 'iso' for older
#    isotropic-only checkpoints. Ignored when MODEL_FAMILY is already the
#    explicit 'gaussian_aniso' / 'gaussian_iso'.
GAUSSIAN_VARIANT = 'aniso'   # 'aniso' or 'iso'

# ── Anisotropic Gaussian V_theta: low-rank precision correction rank ──
#    Sigma_k^{-1} = diag(a_k) + B_k @ B_k^T,  B_k in R^{d x ANISO_RANK}.
#    Only used when the aniso variant is selected. Must match the rank the
#    checkpoint was trained with, or the state_dict load below will report
#    a B_proj shape mismatch.
ANISO_RANK = 4

# ── Override model config (only used if MODEL_FAMILY != 'auto') ──
D              = 256
L              = 8
N_REGISTERS    = 16
V_HIDDEN       = 1024
V_DEPTH        = 3
XI_CHANNELS    = 4
XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]
BLOCK_SIZE     = 512
VOCAB_SIZE     = 50257
MAX_LEN        = 1024

# ── Gaussian V_theta overrides ──
V_THETA_N_HEADS        = 4
V_THETA_WELLS_PER_HEAD = 8

# ── SQ3 V_theta overrides ──
K_MIX    = 8
SQ3_TAU  = 1.0

# ── Data ──
DATASET = 'tinystories'   # 'tinystories' or 'openwebtext'
MAX_TRAIN_TOKENS = 5_000_000

# ── SCAF audit parameters ──
SCAF_SEQ_LEN     = 128
SCAF_N_SEQS      = 16
SCAF_N_TARGETS   = 64
SCAF_MICRO_BATCH = 4
SCAF_SEED        = 0

# ── Leak frame parameters ──
FRAME_N_SEQS     = 16
FRAME_N_PAIRS    = 4
FRAME_SPLITS     = (0.25, 0.5, 0.75)

# ── Legacy probe parameters ──
LEGACY_PROBE_N_PAIRS = 4
LEGACY_HONEST_K      = 256

# ── Register diagnostics ──
DIAG_BATCHES  = 10
DIAG_BATCH_SZ = 4

# ── DoWhy (Phase 6) ──
RUN_DOWHY = True

print(f'Checkpoint:  {CHECKPOINT_PATH}')
print(f'Output:      {OUTPUT_DIR}')
print(f'Model:       {MODEL_FAMILY}  (gaussian_variant={GAUSSIAN_VARIANT}, aniso_rank={ANISO_RANK})')
print(f'Dataset:     {DATASET}')


In [ ]:
# ── Cell 1: Environment + Drive Mount ─────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, copy
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
SCAF_URL    = 'https://github.com/dimitarpg13/semsimula-scaf.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _sh('pip install -q transformers huggingface_hub pyarrow matplotlib seaborn')
    _sh(f'pip install -q "git+{SCAF_URL}#egg=semsimula-scaf[pywhy,plot]"')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

OUT_PATH = Path(OUTPUT_DIR)
OUT_PATH.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'scaleup/debug',
            'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'OUT_PATH  = {OUT_PATH}')


In [ ]:
# ── Cell 2: GPU + Imports ─────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for autograd.grad stability')
else:
    print('WARNING: No GPU detected. Analysis will run on CPU (slow).')

import scaf
print(f'SCAF version: {scaf.__version__ if hasattr(scaf, "__version__") else "dev"}')

from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig

print('Imports OK')


In [ ]:
# ── Cell 3: Load Validation Data ──────────────────────────────────
from data_module import get_batch

if DATASET == 'tinystories':
    from data_module import load_tiny_stories
    train_ids, val_ids = load_tiny_stories(
        n_train_files=1, val_frac=0.01, max_train_tokens=MAX_TRAIN_TOKENS)
elif DATASET == 'openwebtext':
    from data_module import load_openwebtext
    train_ids, val_ids = load_openwebtext()
else:
    raise ValueError(f'Unknown dataset: {DATASET}')

print(f'Dataset: {DATASET}')
print(f'  train: {len(train_ids):,}   val: {len(val_ids):,}')

val_tokens = torch.from_numpy(val_ids[:SCAF_N_SEQS * SCAF_SEQ_LEN].astype(np.int64))
val_tokens = val_tokens.reshape(SCAF_N_SEQS, SCAF_SEQ_LEN)
print(f'  SCAF corpus: {val_tokens.shape}')


In [ ]:
# ── Cell 4: Reconstruct Model from Checkpoint ────────────────────
from dataclasses import asdict
import math as _math

ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
print(f'Checkpoint loaded: {Path(CHECKPOINT_PATH).name}')
print(f'  Keys: {sorted(ckpt.keys())}')

# ── Determine model family ──
detected_family = MODEL_FAMILY
if MODEL_FAMILY == 'auto':
    if 'model_cfg' in ckpt:
        cfg_dict = ckpt['model_cfg']
        print(f'  model_cfg found in checkpoint (d={cfg_dict.get("d")}, L={cfg_dict.get("L")})')
        detected_family = 'from_cfg'
    elif 'v_theta_variant' in ckpt:
        detected_family = ckpt['v_theta_variant']
        print(f'  v_theta_variant from checkpoint: {detected_family}')
    elif 'v_theta_kind' in ckpt:
        detected_family = ckpt['v_theta_kind']
        print(f'  v_theta_kind from checkpoint: {detected_family}')
    else:
        detected_family = 'mlp'
        print(f'  No model_cfg in checkpoint — defaulting to: {detected_family}')

# ── Build model ──
if detected_family == 'from_cfg':
    cfg_dict = dict(ckpt['model_cfg'])
    logfreq_path = cfg_dict.get('logfreq_path', '')
    if logfreq_path and not Path(logfreq_path).exists():
        local_lf = CA_DIR / 'scaleup' / 'results' / Path(logfreq_path).name
        if local_lf.exists():
            cfg_dict['logfreq_path'] = str(local_lf)
        else:
            counts = np.bincount(train_ids.astype(np.int64),
                                 minlength=cfg_dict.get('vocab_size', VOCAB_SIZE)).astype(np.float64)
            p = (counts + 1.0) / (counts.sum() + cfg_dict.get('vocab_size', VOCAB_SIZE))
            surprisal = (-np.log(p)).astype(np.float32)
            _lf_tmp = Path('/tmp/logfreq_scaf.npy')
            np.save(str(_lf_tmp), surprisal)
            cfg_dict['logfreq_path'] = str(_lf_tmp)
            print(f'  Computed logfreq surprisal -> {_lf_tmp}')

    model_cfg = FockMultiXiPARFConfig(**cfg_dict)
    model = FockMultiXiPARFLM(model_cfg)
    print(f'  Built model from checkpoint config (d={model_cfg.d}, L={model_cfg.L})')

else:
    logfreq_path = CA_DIR / 'scaleup' / 'results'
    lf_file = logfreq_path / f'logfreq_surprisal_{DATASET}.npy'
    if not lf_file.exists():
        counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
        p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
        surprisal = (-np.log(p)).astype(np.float32)
        np.save(str(lf_file), surprisal)
        print(f'  Computed logfreq -> {lf_file}')

    model_cfg = FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN, L=L,
        v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=1.0,
        mass_mode='logfreq', logfreq_path=str(lf_file),
        logfreq_init_alpha=0.1, init_gamma=1.0,
        fixed_gamma=ckpt.get('gamma', ckpt.get('fixed_gamma', 0.3)),
        causal_force=True, ln_after_step=True,
        xi_channels=XI_CHANNELS, xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True, xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive',
        v_phi_phi_hidden=128, v_phi_theta_hidden=128,
        top_k=8, v_phi_n_heads=1, score_head_hidden=32,
        gumbel_tau_init=1.0, gumbel_tau_min=0.3, gumbel_noise=True,
        use_gathered_v_phi=True, use_layer_checkpoint=True,
        ln_before_distance=True, per_layer_v_phi_scale=True,
        use_output_bias=False, tie_embeddings=True,
        fock_version='v2', n_registers=N_REGISTERS,
        reverse_channel=True,
        d_k=64, tau_create_init=8.0, creation_gate_hidden=64,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True,
        stack_discipline=True, prefix_causal_registers=True,
    )
    model = FockMultiXiPARFLM(model_cfg)

    if detected_family in ('gaussian', 'gaussian_aniso', 'gaussian_iso'):
        # Explicit checkpoint tags win outright; the bare 'gaussian' tag is
        # ambiguous (older checkpoints/metadata don't distinguish variants)
        # and falls back to the configured default (GAUSSIAN_VARIANT).
        if detected_family == 'gaussian_iso':
            gaussian_variant = 'iso'
        elif detected_family == 'gaussian_aniso':
            gaussian_variant = 'aniso'
        else:
            gaussian_variant = GAUSSIAN_VARIANT

        if gaussian_variant == 'aniso':
            from model_aniso_gaussian_vtheta import (
                AnisotropicDepthConditionedGaussianVTheta,
                install_aniso_depth_routing)
            model.V_theta = AnisotropicDepthConditionedGaussianVTheta(
                d=D, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS, n_layers=L,
                rank=ANISO_RANK, w_scale=1.0, init_log_precision=-_math.log(D),
                precision_max=2.0 / D, code_init_std=0.02,
            )
            install_aniso_depth_routing(model)
            print(f'  V_theta: Anisotropic Gaussian {V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}, '
                  f'rank={ANISO_RANK}')
        else:
            from model_gaussian_vtheta import (
                DepthConditionedMultiContextGaussianVTheta, install_depth_routing)
            model.V_theta = DepthConditionedMultiContextGaussianVTheta(
                d=D, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS, n_layers=L,
                w_scale=1.0, init_log_precision=-_math.log(D),
                precision_max=2.0 / D, code_init_std=0.02,
            )
            install_depth_routing(model)
            print(f'  V_theta: Isotropic Gaussian {V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}')

    elif detected_family == 'sq3':
        from model_structured_vtheta import MixtureQuadraticVTheta
        xi_d = XI_CHANNELS * D
        inner = MixtureQuadraticVTheta.__new__(MixtureQuadraticVTheta)
        torch.nn.Module.__init__(inner)
        inner.d = D; inner.K = K_MIX; inner.tau = SQ3_TAU
        inner.mu_proj = nn.Linear(xi_d, K_MIX * D)
        inner.a_proj  = nn.Linear(xi_d, K_MIX * D)
        inner.pi_proj = nn.Linear(xi_d, K_MIX)
        inner.b_proj  = nn.Linear(xi_d, 1)
        inner._init_weights(0.0)

        class _Adapter(nn.Module):
            def __init__(self, inner, K, d):
                super().__init__()
                self.inner = inner; self.K = K; self.d = d
            def forward(self, xis, h):
                return self.inner(xis.reshape(*xis.shape[:2], -1), h)

        model.V_theta = _Adapter(inner, K=XI_CHANNELS, d=D)
        print(f'  V_theta: SQ3 Mixture K_mix={K_MIX}')

    else:
        print(f'  V_theta: MLP (default)')

# ── Load weights ──
missing, unexpected = model.load_state_dict(ckpt['model_state_dict'], strict=False)
if missing:
    print(f'  Missing keys:    {missing}')
if unexpected:
    print(f'  Unexpected keys: {unexpected}')

# Isotropic <-> anisotropic mismatch is the most common cause of missing/
# unexpected V_theta keys: 'B_proj' only exists on the anisotropic bank's
# low-rank precision factor. Surface this immediately rather than letting
# it manifest as silently-wrong wells downstream.
_b_proj_missing = any('B_proj' in k for k in missing)
_b_proj_unexpected = any('B_proj' in k for k in unexpected)
if _b_proj_missing:
    print('  WARNING: checkpoint has no B_proj weights but the model was '
          "built with the anisotropic V_theta. Set GAUSSIAN_VARIANT = 'iso' "
          '(or MODEL_FAMILY = \'gaussian_iso\') and re-run.')
elif _b_proj_unexpected:
    print("  WARNING: checkpoint has B_proj weights but the model was built "
          "with the isotropic V_theta. Set GAUSSIAN_VARIANT = 'aniso' "
          "(or MODEL_FAMILY = 'gaussian_aniso') and re-run.")

model.to(DEVICE).eval()
n_params = sum(p.numel() for p in model.parameters())
n_vt = sum(p.numel() for p in model.V_theta.parameters())
print(f'  Total params:  {n_params:,}')
print(f'  V_theta params: {n_vt:,}')
print(f'  gamma: {model.gamma.item():.3f}')

# ── Checkpoint metadata ──
ckpt_meta = {}
for k in ['step', 'val_ppl', 'best_val_ppl', 'val_loss', 'gamma',
          'variant', 'experiment', 'tag', 'seed', 'v_theta_kind',
          'v_theta_variant', 'K_mix']:
    if k in ckpt:
        ckpt_meta[k] = ckpt[k]
        print(f'  ckpt[{k}] = {ckpt[k]}')

del ckpt
gc.collect()
print('Model loaded OK')


## Phase 1 — SCAF Full Audit

Runs the complete SCAF audit battery:
- **Controls**: determinism, placebo, positive
- **Probes**: future perturbation ($\ell_\infty$ logit deviation), target relocation (honest PPL gap)
- **Diagnostics**: mediation attribution (if leak found)


In [ ]:
# ── Cell 5: Phase 1 — SCAF Full Audit ─────────────────────────────

print('=' * 64)
print('  PHASE 1: SCAF Full Audit')
print('=' * 64)

scorecard = scaf.audit(
    model,
    tokens=val_tokens,
    device=DEVICE,
    dtype='float32',
    seq_len=SCAF_SEQ_LEN,
    n_seqs=SCAF_N_SEQS,
    n_targets=SCAF_N_TARGETS,
    micro_batch=SCAF_MICRO_BATCH,
    seed=SCAF_SEED,
    mediation=True,
)

print(scorecard.summary())

sc_path = OUT_PATH / 'phase1_scaf_audit.json'
with open(str(sc_path), 'w') as f:
    json.dump(scorecard.to_dict(), f, indent=2, default=str)
print(f'Saved: {sc_path}')


## Phase 1.5 — Geometric Leak Probes (Tier A / Tier B)

Runs the SCAF **geometric** diagnostics directly against the loaded
checkpoint. These operate in the model's internal hidden-state geometry
rather than on logits, and run as *diagnostics* — they add a second,
structural axis of evidence on top of Phase 1's verdict without changing it.

- **Tier A — `HiddenStateLeakProbe`**: per-layer cosine deviation of hidden
  states under `do(future)`. Catches *latent* leaks that have corrupted the
  hidden state but haven't reached the logits yet (`latent_leak=True`).
- **Tier B — `BasinMembershipProbe`**: whether the future perturbation flips
  a past hidden state's dominant attractor well in the Gaussian $V_\theta$
  landscape — a discrete, structurally more severe signal than continuous
  cosine deviation.

Availability depends on the checkpoint's `V_theta` family:

| `V_theta` family | `has_hidden_states` | `has_vtheta_wells` |
|---|:---:|:---:|
| MLP (`ScalarPotentialMultiXi`) | ✅ | ❌ (skips loudly) |
| Isotropic Gaussian (`MixtureGaussianVTheta`) | ✅ | ✅ |
| Anisotropic Gaussian (`Anisotropic*GaussianVTheta`) | ✅ | ✅ |

Both isotropic and anisotropic Gaussian families work with Tier B: the
adapter normalises the isotropic 3-tuple `_components()` return
(`mu, a, w`) to a rank-0 low-rank factor, which is mathematically identical
to having no low-rank correction at all.

Model reconstruction (Cell 4) defaults to the **anisotropic** variant
(`GAUSSIAN_VARIANT = 'aniso'`) whenever the family resolves to the ambiguous
`'gaussian'` tag, since that's the most likely family for d384/d768
checkpoints. Set `GAUSSIAN_VARIANT = 'iso'` (or `MODEL_FAMILY =
'gaussian_iso'` / `'gaussian_aniso'` explicitly) if reconstruction picks the
wrong one — a `B_proj`-related missing/unexpected-key warning at load time
is the tell.

In [ ]:
# ── Cell 5b: Phase 1.5 — Geometric Leak Probes (Tier A/B) ────────

print('=' * 64)
print('  PHASE 1.5: Geometric Leak Probes (Tier A/B)')
print('=' * 64)

corpus_geo = scaf.TokenCorpus(val_tokens, seq_len=SCAF_SEQ_LEN, seed=SCAF_SEED)

with scaf.InterventableModel(model, device=DEVICE, dtype='float32') as im:
    print(f'  adapter:           {im.adapter.name}')
    print(f'  has_hidden_states: {im.caps.has_hidden_states}')
    print(f'  has_vtheta_wells:  {im.caps.has_vtheta_wells}')

    tier_a = None
    if im.caps.has_hidden_states:
        tier_a = scaf.HiddenStateLeakProbe(
            splits=FRAME_SPLITS, n_seqs=SCAF_N_SEQS, n_pairs=2,
            micro_batch=SCAF_MICRO_BATCH,
        ).run(im, corpus_geo)
        print(f'\n  [Tier A] {tier_a}')
        if not tier_a.skipped:
            per_layer_a = tier_a.detail['per_layer_delta_cos']
            print(f'    per-layer dcos: {[f"{v:.4e}" for v in per_layer_a]}')
            print(f'    peak layer:     {tier_a.detail["peak_layer"]}')
            print(f'    latent_leak:    {tier_a.detail["latent_leak"]}')
    else:
        print('\n  [Tier A] SKIPPED -- adapter does not expose hidden states')

    tier_b = None
    if im.caps.has_vtheta_wells:
        tier_b = scaf.BasinMembershipProbe(
            splits=FRAME_SPLITS, n_seqs=SCAF_N_SEQS, n_pairs=2,
            micro_batch=SCAF_MICRO_BATCH,
        ).run(im, corpus_geo)
        print(f'\n  [Tier B] {tier_b}')
        if not tier_b.skipped:
            per_layer_b = tier_b.detail['per_layer_crossing_rate']
            print(f'    per-layer beta: {[f"{v:.4e}" for v in per_layer_b]}')
            print(f'    worst layer:    {tier_b.detail["worst_layer"]}')
    else:
        print('\n  [Tier B] SKIPPED -- V_theta does not expose well '
              'parameters (expected for the MLP V_theta family)')

    geo_result = {
        'has_hidden_states': bool(im.caps.has_hidden_states),
        'has_vtheta_wells': bool(im.caps.has_vtheta_wells),
        'tier_a': tier_a.to_dict() if tier_a is not None else None,
        'tier_b': tier_b.to_dict() if tier_b is not None else None,
    }

geo_path = OUT_PATH / 'phase1_5_geometric_probes.json'
with open(str(geo_path), 'w') as f:
    json.dump(geo_result, f, indent=2, default=str)
print(f'\nSaved: {geo_path}')

## Phase 2 — Interventional Leak Frame

Builds a tidy counterfactual dataset where each row is a scored target
position under factual or $\text{do}(\text{future} := \text{resampled})$.
The ATE is computed with exact paired sign-flip inference.


In [ ]:
# ── Cell 6: Phase 2 — Interventional Leak Frame ──────────────────

print('=' * 64)
print('  PHASE 2: Interventional Leak Frame')
print('=' * 64)

frame = scaf.build_leak_frame(
    model,
    tokens=val_tokens,
    device=DEVICE,
    dtype='float32',
    seq_len=SCAF_SEQ_LEN,
    n_seqs=FRAME_N_SEQS,
    n_pairs=FRAME_N_PAIRS,
    splits=FRAME_SPLITS,
    micro_batch=SCAF_MICRO_BATCH,
    seed=SCAF_SEED,
)

print(frame.summary(test=True))

ate_result = frame.ate_test()
print(f'\nATE = {ate_result["ate"]:+.6f} nats')
print(f'  p-value = {ate_result["p_value"]:.4e}')
print(f'  95% CI  = [{ate_result["ci"][0]:+.6f}, {ate_result["ci"][1]:+.6f}]')
print(f'  n_units = {ate_result["n_units"]}')

profile = frame.ate_by('distance_to_cut')
print(f'\nLeak profile by distance_to_cut:')
for dist, ate in sorted(profile.items()):
    print(f'  distance={dist:3d}  ATE={ate:+.6f} nats')

frame_path = OUT_PATH / 'phase2_leak_frame.csv'
frame.to_csv(str(frame_path))
print(f'\nSaved: {frame_path}')

ate_path = OUT_PATH / 'phase2_ate_result.json'
with open(str(ate_path), 'w') as f:
    json.dump({k: (list(v) if isinstance(v, tuple) else v)
               for k, v in ate_result.items()}, f, indent=2, default=str)
print(f'Saved: {ate_path}')


## Phase 2.5 — Geometric Leak Frame (CATE by layer / basin)

Builds a **separate** counterfactual dataset with `include_hidden_states=True`
and `include_basin_membership=True`. Each row is now keyed by
`(layer, position)` rather than `position` alone, which is why this is kept
out of Phase 2's exact-ATE frame — that frame's paired sign-flip test assumes
one row per scored position, and multiplying by layer count would understate
the standard error (pseudo-replication). Use this frame only for the
layer/basin breakdown, not as a substitute for Phase 2's headline ATE.

Kept intentionally small (`n_seqs`, `n_pairs` capped below the Phase 2
defaults) — this frame's row count scales with `n_layers`, which gets
expensive fast.

In [ ]:
# ── Cell 6b: Phase 2.5 — Geometric Leak Frame ────────────────────

print('=' * 64)
print('  PHASE 2.5: Geometric Leak Frame (CATE by layer / basin)')
print('=' * 64)

GEO_FRAME_N_SEQS  = min(FRAME_N_SEQS, 8)
GEO_FRAME_N_PAIRS = min(FRAME_N_PAIRS, 2)

geo_frame = scaf.build_leak_frame(
    model,
    tokens=val_tokens,
    device=DEVICE,
    dtype='float32',
    seq_len=SCAF_SEQ_LEN,
    n_seqs=GEO_FRAME_N_SEQS,
    n_pairs=GEO_FRAME_N_PAIRS,
    splits=FRAME_SPLITS,
    micro_batch=SCAF_MICRO_BATCH,
    seed=SCAF_SEED,
    include_hidden_states=True,
    include_basin_membership=True,
)

print(geo_frame.summary(test=False))
print(f'  columns: {list(geo_frame.columns.keys())}')

layer_col = np.array(geo_frame.column('layer'))
arm_col = np.array(geo_frame.column('future_perturbed'))
cf_mask = arm_col == 1  # deviation is 0 by construction on the factual arm

print('\n  Mean cosine deviation by layer (counterfactual arm):')
dev_col = np.array(geo_frame.column('hidden_cos_dev'))
for ell in sorted(set(layer_col.tolist())):
    m = cf_mask & (layer_col == ell)
    if m.any():
        print(f'    layer {ell:2d}: {dev_col[m].mean():.4e}')

if 'basin_changed' in geo_frame.columns:
    print('\n  Basin-crossing rate by layer (counterfactual arm):')
    basin_col = np.array(geo_frame.column('basin_changed'))
    for ell in sorted(set(layer_col.tolist())):
        m = cf_mask & (layer_col == ell)
        if m.any():
            print(f'    layer {ell:2d}: {basin_col[m].mean():.4e}')
else:
    print('\n  basin_changed column absent -- checkpoint V_theta has no '
        'well parameters (expected for the MLP V_theta family)')

geo_frame_path = OUT_PATH / 'phase2_5_geometric_leak_frame.csv'
geo_frame.to_csv(str(geo_frame_path))
print(f'\nSaved: {geo_frame_path}')

## Phase 3 — Legacy Trained-Scale Leak Probe

The original two-part probe from the training notebooks:
1. **Future perturbation** at trained scale (max logit deviation, mean dNLL)
2. **Honest vs standard PPL** on the same target tokens


In [ ]:
# ── Cell 7: Phase 3 — Legacy Trained Leak Probe ──────────────────
from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

print('=' * 64)
print('  PHASE 3: Legacy Trained-Scale Leak Probe')
print('=' * 64)

probe_res = probe_trained_leak(
    model, val_ids, device=DEVICE, context=BLOCK_SIZE,
    n_pairs=LEGACY_PROBE_N_PAIRS, use_float64=False)

print(f'\n  max|dlogit|(past) = {probe_res["max_dlogit_past"]:.3e}')
print(f'  mean dNLL(past)   = {probe_res["mean_dnll_past"]:+.6f} nats')
print(f'  gate zero control = {probe_res["gate_zero_control"]:.3e}')

honest_res = honest_ppl_test(
    model, val_ids, k=LEGACY_HONEST_K,
    context=BLOCK_SIZE, batch=DIAG_BATCH_SZ, device=DEVICE)

print(f'\n  Standard PPL (mid-window): {honest_res["ppl_mid_window"]:.2f}')
print(f'  Honest PPL (last-pos):     {honest_res["ppl_last_pos"]:.2f}')
print(f'  Paired diff:               {honest_res["paired_diff_nats"]:+.6f} '
      f'+/- {honest_res["paired_diff_se"]:.6f} nats')

leak_status = 'CLEAN' if honest_res['paired_diff_nats'] < 0.1 else 'LEAK'
print(f'  Verdict: [{leak_status}]')

model.train()
model.eval()

legacy_result = {
    'probe': probe_res,
    'honest_ppl': honest_res,
    'verdict': leak_status,
}
legacy_path = OUT_PATH / 'phase3_legacy_probe.json'
with open(str(legacy_path), 'w') as f:
    json.dump(legacy_result, f, indent=2, default=str)
print(f'\nSaved: {legacy_path}')


## Phase 4 — Register Diagnostics

Per-layer routing quality metrics:
- **Normalised attention entropy** — 0 = peaked routing, 1 = mean-pool
- **Register content diversity** — 0 = identical, 1 = orthogonal
- **$\alpha_{\max}$** — max creation gate attention weight


In [ ]:
# ── Cell 8: Phase 4 — Register Diagnostics ───────────────────────

print('=' * 64)
print('  PHASE 4: Register Diagnostics')
print('=' * 64)

model.eval()

entropy_per_layer = []
diversity_per_layer = []
alpha_max_per_layer = []

rng_diag = np.random.default_rng(42)

for bi in range(DIAG_BATCHES):
    x_np, _ = get_batch(val_ids, DIAG_BATCH_SZ, BLOCK_SIZE, rng_diag)
    x = torch.from_numpy(x_np).to(DEVICE)

    with torch.enable_grad():
        h0 = model._embed(x)
        h_L, _ = model._stack_forward(h0, x, return_trajectory=False)

    for ell, fock_layer in enumerate(model.fock_layers):
        if not hasattr(fock_layer, '_last_creation_alpha'):
            continue
        alpha = fock_layer._last_creation_alpha
        if alpha is None:
            continue

        if alpha.dim() == 4:
            alpha_2d = alpha[:, :, :, 0]
        elif alpha.dim() == 3:
            alpha_2d = alpha
        else:
            continue

        eps = 1e-12
        log_a = torch.log(alpha_2d.clamp(min=eps))
        T_dim = alpha_2d.shape[1]
        H = -(alpha_2d * log_a).sum(dim=1) / max(np.log(T_dim), 1.0)
        entropy_val = H.mean().item()

        amax = alpha_2d.max(dim=1).values.mean().item()

        if hasattr(fock_layer, '_last_r_new_content'):
            r = fock_layer._last_r_new_content
            if r is not None and r.dim() >= 3:
                M_dim = r.shape[-2] if r.dim() == 3 else r.shape[-2]
                r_flat = r.reshape(-1, M_dim, r.shape[-1])
                r_norm = F.normalize(r_flat, dim=-1)
                cos_sim = torch.bmm(r_norm, r_norm.transpose(1, 2))
                mask = 1.0 - torch.eye(M_dim, device=cos_sim.device)
                off_diag = (cos_sim * mask).sum(dim=(1, 2)) / max(M_dim * (M_dim - 1), 1)
                div_val = 1.0 - off_diag.mean().item()
            else:
                div_val = float('nan')
        else:
            div_val = float('nan')

        while len(entropy_per_layer) <= ell:
            entropy_per_layer.append([])
            diversity_per_layer.append([])
            alpha_max_per_layer.append([])

        entropy_per_layer[ell].append(entropy_val)
        diversity_per_layer[ell].append(div_val)
        alpha_max_per_layer[ell].append(amax)

    del x, h0, h_L
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

n_layers_diag = len(entropy_per_layer)
diag_summary = {}
print(f'\n{"Layer":>6}  {"Entropy":>8}  {"alpha_max":>9}  {"Diversity":>9}')
print('-' * 40)
for ell in range(n_layers_diag):
    e = np.mean(entropy_per_layer[ell]) if entropy_per_layer[ell] else float('nan')
    a = np.mean(alpha_max_per_layer[ell]) if alpha_max_per_layer[ell] else float('nan')
    d = np.mean(diversity_per_layer[ell]) if diversity_per_layer[ell] else float('nan')
    diag_summary[ell] = {'entropy': e, 'alpha_max': a, 'diversity': d}
    print(f'{ell:6d}  {e:8.4f}  {a:9.4f}  {d:9.4f}')

mean_entropy = np.nanmean([v['entropy'] for v in diag_summary.values()])
mean_amax = np.nanmean([v['alpha_max'] for v in diag_summary.values()])
mean_div = np.nanmean([v['diversity'] for v in diag_summary.values()])
print(f'\nOverall: entropy={mean_entropy:.4f}  alpha_max={mean_amax:.4f}  diversity={mean_div:.4f}')

if mean_div >= 0.6 and 0.1 <= mean_entropy <= 0.5:
    routing_verdict = 'ROUTING'
elif mean_div < 0.3 and mean_entropy > 0.8:
    routing_verdict = 'MEAN-POOL'
else:
    routing_verdict = 'MIXED'
print(f'VERDICT: {routing_verdict}')

diag_path = OUT_PATH / 'phase4_register_diagnostics.json'
with open(str(diag_path), 'w') as f:
    json.dump({'per_layer': {str(k): v for k, v in diag_summary.items()},
               'overall': {'entropy': mean_entropy, 'alpha_max': mean_amax,
                           'diversity': mean_div, 'verdict': routing_verdict}},
              f, indent=2)
print(f'Saved: {diag_path}')


## Phase 5 — Visualisation Dashboard

Six-panel summary figure:
1. SCAF audit scorecard (text)
2. Within-window NLL profile (bar chart)
3. Leak profile by distance-to-cut (CATE)
4. Per-layer register entropy (bar)
5. Per-layer register diversity (bar)
6. Per-layer alpha\_max (bar)


In [ ]:
# ── Cell 9: Phase 5 — Visualisation Dashboard ────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(3, 2, hspace=0.4, wspace=0.3)

# ── Panel 1: Audit Scorecard (text) ──
ax1 = fig.add_subplot(gs[0, 0])
ax1.axis('off')

verdict = scorecard.verdict
verdict_color = {'CLEAN': '#2ecc71', 'LEAK': '#e74c3c', 'INVALID': '#f39c12'}[verdict]

summary_lines = []
summary_lines.append(f'SCAF Audit Verdict: {verdict}')
summary_lines.append('')
for cr in scorecard.controls:
    summary_lines.append(f'  {cr.status:4s}  {cr.name}: {cr.statistic:.3e} {cr.unit}')
for pr in scorecard.probes:
    summary_lines.append(f'  {pr.status:4s}  {pr.name}: {pr.statistic:.3e} {pr.unit}')
if scorecard.diagnostics:
    summary_lines.append('')
    for dr in scorecard.diagnostics:
        summary_lines.append(f'  {dr.status:4s}  {dr.name}: {dr.statistic:.3f} {dr.unit}')

tr = scorecard.get('target_relocation')
if tr and tr.detail:
    summary_lines.append('')
    summary_lines.append(f'  Standard PPL: {tr.detail.get("ppl_standard", "?"):.2f}')
    summary_lines.append(f'  Honest PPL:   {tr.detail.get("ppl_honest", "?"):.2f}')

summary_text = '\n'.join(summary_lines)
ax1.text(0.05, 0.95, summary_text, transform=ax1.transAxes, fontsize=9,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor=verdict_color, alpha=0.15))
ax1.set_title('SCAF Audit Scorecard', fontsize=12, fontweight='bold')

# ── Panel 2: Within-Window NLL Profile ──
ax2 = fig.add_subplot(gs[0, 1])
if 'position_profile' in honest_res and honest_res['position_profile']:
    profile_data = honest_res['position_profile']
    positions = list(range(len(profile_data)))
    ax2.bar(positions, profile_data, color='steelblue', alpha=0.7, width=1.0)
    ax2.set_xlabel('Position in window')
    ax2.set_ylabel('Mean NLL (nats)')
    ax2.set_title('Within-Window NLL Profile', fontsize=12, fontweight='bold')
    ax2.axhline(y=np.mean(profile_data), color='red', linestyle='--', alpha=0.5, label='mean')
    ax2.legend(fontsize=8)
else:
    ax2.text(0.5, 0.5, 'No position profile available', transform=ax2.transAxes,
             ha='center', va='center', fontsize=12)
    ax2.set_title('Within-Window NLL Profile', fontsize=12, fontweight='bold')

# ── Panel 3: Leak Profile by Distance to Cut ──
ax3 = fig.add_subplot(gs[1, 0])
if profile:
    dists = sorted(profile.keys())
    ates = [profile[d] for d in dists]
    colors = ['#e74c3c' if a > 0.01 else '#2ecc71' for a in ates]
    ax3.bar(dists, ates, color=colors, alpha=0.7)
    ax3.axhline(y=0, color='black', linewidth=0.5)
    ax3.set_xlabel('Distance to cut (positions)')
    ax3.set_ylabel('ATE (nats)')
    ax3.set_title('Leak Profile: CATE by Distance to Cut', fontsize=12, fontweight='bold')
else:
    ax3.text(0.5, 0.5, 'No leak frame data', transform=ax3.transAxes,
             ha='center', va='center', fontsize=12)
    ax3.set_title('Leak Profile', fontsize=12, fontweight='bold')

# ── Panel 4: Per-Layer Entropy ──
ax4 = fig.add_subplot(gs[1, 1])
if diag_summary:
    layers = sorted(diag_summary.keys())
    entropies = [diag_summary[l]['entropy'] for l in layers]
    ax4.bar(layers, entropies, color='#3498db', alpha=0.7)
    ax4.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='routing threshold')
    ax4.axhline(y=0.8, color='orange', linestyle='--', alpha=0.5, label='mean-pool threshold')
    ax4.set_xlabel('Layer')
    ax4.set_ylabel('Normalised Entropy')
    ax4.set_title('Register Attention Entropy', fontsize=12, fontweight='bold')
    ax4.legend(fontsize=8)
    ax4.set_ylim(0, 1.1)

# ── Panel 5: Per-Layer Diversity ──
ax5 = fig.add_subplot(gs[2, 0])
if diag_summary:
    diversities = [diag_summary[l]['diversity'] for l in layers]
    ax5.bar(layers, diversities, color='#27ae60', alpha=0.7)
    ax5.axhline(y=0.6, color='red', linestyle='--', alpha=0.5, label='routing threshold')
    ax5.set_xlabel('Layer')
    ax5.set_ylabel('Content Diversity')
    ax5.set_title('Register Content Diversity', fontsize=12, fontweight='bold')
    ax5.legend(fontsize=8)
    ax5.set_ylim(0, 1.1)

# ── Panel 6: Per-Layer alpha_max ──
ax6 = fig.add_subplot(gs[2, 1])
if diag_summary:
    amaxes = [diag_summary[l]['alpha_max'] for l in layers]
    ax6.bar(layers, amaxes, color='#e67e22', alpha=0.7)
    ax6.set_xlabel('Layer')
    ax6.set_ylabel('Mean alpha_max')
    ax6.set_title('Creation Gate Attention Strength', fontsize=12, fontweight='bold')
    ax6.set_ylim(0, 1.1)

# ── Save ──
fig.suptitle(f'SCAF Causal Leak Analysis — Fock v2.1 PARFLM\n'
             f'Checkpoint: {Path(CHECKPOINT_PATH).name}  |  Verdict: {verdict}',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()

dashboard_path = OUT_PATH / 'phase5_dashboard.png'
fig.savefig(str(dashboard_path), dpi=150, bbox_inches='tight')
print(f'Saved: {dashboard_path}')
plt.show()

# ── Additional: ATE bar chart ──
fig2, ax_ate = plt.subplots(figsize=(8, 4))
ate_val = ate_result['ate']
ci_lo, ci_hi = ate_result['ci']
color = '#2ecc71' if abs(ate_val) < 0.01 else '#e74c3c'
ax_ate.bar(['ATE'], [ate_val], color=color, alpha=0.7)
ax_ate.errorbar(['ATE'], [ate_val], yerr=[[ate_val - ci_lo], [ci_hi - ate_val]],
                fmt='none', color='black', capsize=10, linewidth=2)
ax_ate.axhline(y=0, color='black', linewidth=0.5)
ax_ate.set_ylabel('Effect (nats)')
ax_ate.set_title(f'Average Treatment Effect: do(future) on NLL\n'
                 f'ATE = {ate_val:+.6f} nats  (p = {ate_result["p_value"]:.4e})',
                 fontsize=11)
fig2.tight_layout()
ate_plot_path = OUT_PATH / 'phase5_ate_bar.png'
fig2.savefig(str(ate_plot_path), dpi=150)
print(f'Saved: {ate_plot_path}')
plt.show()


## Phase 6 — DoWhy/EconML Estimands *(optional)*

Formal causal estimation with:
- ATE via paired sign-flip test + bootstrap CI
- CATE by distance-to-cut via CausalForestDML
- Refutation tests (placebo treatment, random common cause)


In [ ]:
# ── Cell 10: Phase 6 — DoWhy/EconML Estimands ────────────────────

if not RUN_DOWHY:
    print('Phase 6 skipped (RUN_DOWHY = False)')
else:
    print('=' * 64)
    print('  PHASE 6: DoWhy/EconML Estimands')
    print('=' * 64)

    try:
        est_report = scaf.estimate_leak(
            frame,
            outcome='nll_within',
            cate_axes=('distance_to_cut',),
            cate_model='forest',
            refuters=('placebo_treatment_refuter', 'random_common_cause'),
            num_simulations=20,
            dowhy_inference=False,
        )
        print(est_report.summary())

        est_path = OUT_PATH / 'phase6_estimation_report.json'
        with open(str(est_path), 'w') as f:
            report_dict = {
                'ate': est_report.ate,
                'stderr': est_report.stderr,
                'ci': list(est_report.ci) if est_report.ci else None,
                'p_value': est_report.p_value,
                'reference_ate': est_report.reference_ate,
                'agrees_with_reference': est_report.agrees_with_reference,
                'refutations_ok': est_report.refutations_ok,
            }
            json.dump(report_dict, f, indent=2, default=str)
        print(f'Saved: {est_path}')

        # ── CATE plot ──
        if est_report.cate and 'distance_to_cut' in est_report.cate:
            cate_data = est_report.cate['distance_to_cut']
            fig3, ax_cate = plt.subplots(figsize=(10, 4))

            exact_profile = frame.ate_by('distance_to_cut')
            if exact_profile:
                dists = sorted(exact_profile.keys())
                exact_ates = [exact_profile[d] for d in dists]
                ax_cate.bar(dists, exact_ates, color='steelblue', alpha=0.4, label='Exact stratified')

            if isinstance(cate_data, dict):
                forest_dists = sorted(cate_data.keys())
                forest_ates = [cate_data[d] for d in forest_dists]
                ax_cate.plot(forest_dists, forest_ates, 'ro-', markersize=5,
                            label='CausalForestDML', linewidth=2)

            ax_cate.axhline(y=0, color='black', linewidth=0.5)
            ax_cate.set_xlabel('Distance to cut (positions)')
            ax_cate.set_ylabel('CATE (nats)')
            ax_cate.set_title('Heterogeneous Treatment Effect by Distance to Cut')
            ax_cate.legend()
            fig3.tight_layout()

            cate_path = OUT_PATH / 'phase6_cate_by_distance.png'
            fig3.savefig(str(cate_path), dpi=150)
            print(f'Saved: {cate_path}')
            plt.show()

    except ImportError as e:
        print(f'DoWhy/EconML not available: {e}')
        print('Install with: pip install "semsimula-scaf[pywhy]"')
    except Exception as e:
        print(f'Phase 6 error: {e}')
        import traceback
        traceback.print_exc()


In [ ]:
# ── Cell 11: Final Summary ────────────────────────────────────────

print()
print('=' * 64)
print('  SCAF CHECKPOINT ANALYSIS — FINAL SUMMARY')
print('=' * 64)
print()
print(f'Checkpoint: {Path(CHECKPOINT_PATH).name}')
print(f'Model:      Fock v2.1 PARFLM  (d={model_cfg.d}, L={model_cfg.L}, M={model_cfg.n_registers})')
print(f'Device:     {DEVICE}')
print()

print('Phase 1 — SCAF Audit')
print(f'  Verdict: {scorecard.verdict}')
for cr in scorecard.controls:
    print(f'    {cr.status:4s}  {cr.name}: {cr.statistic:.3e}')
for pr in scorecard.probes:
    print(f'    {pr.status:4s}  {pr.name}: {pr.statistic:.3e} {pr.unit}')
print()

print('Phase 2 — Leak Frame')
print(f'  ATE = {ate_result["ate"]:+.6f} nats  (p = {ate_result["p_value"]:.4e})')
print()

print('Phase 3 — Legacy Probe')
print(f'  max|dlogit|(past) = {probe_res["max_dlogit_past"]:.3e}')
print(f'  Honest PPL: {honest_res["ppl_last_pos"]:.2f}  '
      f'Standard PPL: {honest_res["ppl_mid_window"]:.2f}  '
      f'diff: {honest_res["paired_diff_nats"]:+.4f} nats  [{leak_status}]')
print()

print('Phase 4 — Register Diagnostics')
print(f'  Entropy: {mean_entropy:.4f}  Diversity: {mean_div:.4f}  '
      f'alpha_max: {mean_amax:.4f}  [{routing_verdict}]')
print()

print('Phase 1.5 — Geometric Leak Probes (Tier A/B)')
if geo_result['tier_a'] is not None:
    ta = geo_result['tier_a']
    print(f'  [Tier A] max_cos_dev={ta["statistic"]:.4e}  '
          f'peak_layer={ta["detail_peak_layer"]}  '
          f'latent_leak={ta["detail_latent_leak"]}  [{ta["status"]}]')
else:
    print('  [Tier A] SKIPPED (no hidden-state support)')
if geo_result['tier_b'] is not None:
    tb = geo_result['tier_b']
    print(f'  [Tier B] basin_crossing_rate={tb["statistic"]:.4e}  '
          f'worst_layer={tb["detail_worst_layer"]}  [{tb["status"]}]')
else:
    print('  [Tier B] SKIPPED (no V_theta well parameters)')
print()

print(f'All outputs saved to: {OUT_PATH}')
saved_files = sorted(OUT_PATH.glob('phase*'))
for f in saved_files:
    print(f'  {f.name}')
print()

overall_clean = (scorecard.verdict == 'CLEAN'
                 and leak_status == 'CLEAN'
                 and abs(ate_result['ate']) < 0.01)

if overall_clean:
    print('OVERALL ASSESSMENT: Model is CAUSALLY CLEAN')
    print('  No evidence of future information leakage across all analysis phases.')
else:
    print('OVERALL ASSESSMENT: FURTHER INVESTIGATION NEEDED')
    if scorecard.verdict != 'CLEAN':
        print(f'  SCAF audit: {scorecard.verdict}')
    if leak_status != 'CLEAN':
        print(f'  Legacy probe: {leak_status}')
    if abs(ate_result['ate']) >= 0.01:
        print(f'  ATE: {ate_result["ate"]:+.6f} nats')

print()
print('Done!')
